# Query-Sample Sensitivity and Cross-Lingual Rank Transfer in Pairwise LLM-as-a-Judge Evaluation (MIRAGE-Bench)
**Author:** Devarasetty Seetesh (`MRA2026015`)  
**Dataset:** `nthakur/mirage-bench-pairwise-judgments` (18 languages, 19 models)  
**Seeds:** `42` (global/subsampling/bootstrap), `123` (per-model rank tracking)


In [ ]:
# Install required dependencies if running in Google Colab
# !pip install -q datasets pandas numpy scipy matplotlib pyarrow PyYAML

import os
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import kendalltau, spearmanr

GLOBAL_SEED = 42
SUBSAMPLE_SEED = 42
INDIVIDUAL_SEED = 123
np.random.seed(GLOBAL_SEED)

os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
print("Environment initialized successfully.")


## 1. Dataset Loading & Schema Validation
Load the English (`en`) development split and verify canonical schema columns (`query_id`, `model_A`, `model_B`, `verdict`) and verdict labels (`A`, `B`, `Tie`).


In [ ]:
DATASET_ID = "nthakur/mirage-bench-pairwise-judgments"
LANGUAGES = [
    "ar", "bn", "de", "en", "es", "fa", "fi", "fr", "hi",
    "id", "ja", "ko", "ru", "sw", "te", "th", "yo", "zh"
]

def validate_pairwise_df(df: pd.DataFrame) -> pd.DataFrame:
    required = ["query_id", "model_A", "model_B", "verdict"]
    cleaned = df.dropna(subset=required).copy()
    cleaned["verdict"] = cleaned["verdict"].astype(str).str.strip()
    cleaned = cleaned[cleaned["verdict"].isin(["A", "B", "Tie"])].copy()
    return cleaned

en_raw = load_dataset(DATASET_ID, "en")["dev"].to_pandas()
en_df = validate_pairwise_df(en_raw)

print(f"Validated English records: {len(en_df)}")
print(f"Unique queries: {en_df['query_id'].nunique()}")
print(f"Unique models: {len(set(en_df['model_A']).union(set(en_df['model_B'])))}")
print("Verdict counts:\n", en_df["verdict"].value_counts())


## 2. Method 1 (M1): Baseline Pairwise Win-Rate Ranking
Compute $\text{WinRate}(m) = \frac{W_m + 0.5 T_m}{N_m}$ across all 19 models on English (`en`, `dev`).


In [ ]:
def calculate_model_ranking(data: pd.DataFrame, tie_weight: float = 0.5) -> pd.DataFrame:
    stats = defaultdict(lambda: {"wins": 0, "losses": 0, "ties": 0, "comparisons": 0})
    for _, row in data.iterrows():
        a, b, verdict = row["model_A"], row["model_B"], row["verdict"]
        stats[a]["comparisons"] += 1
        stats[b]["comparisons"] += 1
        if verdict == "A":
            stats[a]["wins"] += 1
            stats[b]["losses"] += 1
        elif verdict == "B":
            stats[b]["wins"] += 1
            stats[a]["losses"] += 1
        elif verdict == "Tie":
            stats[a]["ties"] += 1
            stats[b]["ties"] += 1

    ranking = pd.DataFrame.from_dict(stats, orient="index")
    ranking.index.name = "model"
    ranking["win_rate"] = (ranking["wins"] + tie_weight * ranking["ties"]) / ranking["comparisons"]
    ranking = ranking.sort_values(by="win_rate", ascending=False)
    ranking["rank"] = range(1, len(ranking) + 1)
    return ranking

en_ranking = calculate_model_ranking(en_df)
en_ranking.to_csv("results/tables/model_ranking.csv")
display(en_ranking)


## 3. Method 2 (M2): Query-Sample Sensitivity & Tier Volatility
Perform query-grouped subsampling (`R=100` repeats without replacement) across `25%`, `50%`, `75%`, and `100%` query budgets and compute Kendall's $\tau_b$, Spearman's $\rho$, empirical $95\%$ intervals, and per-model rank standard deviations.


In [ ]:
fractions = [0.25, 0.50, 0.75, 1.00]
n_repeats = 100
rng = np.random.default_rng(SUBSAMPLE_SEED)
query_ids = en_df["query_id"].unique()
ref_scores = en_ranking["win_rate"]

records = []
for frac in fractions:
    k = max(1, int(len(query_ids) * frac))
    for r in range(n_repeats):
        sampled_q = query_ids if frac == 1.0 else rng.choice(query_ids, size=k, replace=False)
        sub_df = en_df[en_df["query_id"].isin(sampled_q)]
        sub_rank = calculate_model_ranking(sub_df)
        common = sorted(set(ref_scores.index) & set(sub_rank.index))
        tau = kendalltau([ref_scores[m] for m in common], [sub_rank.loc[m, "win_rate"] for m in common], variant="b").statistic
        rho = spearmanr([ref_scores[m] for m in common], [sub_rank.loc[m, "win_rate"] for m in common]).statistic
        records.append({"fraction": frac, "queries_used": k, "repeat": r + 1, "kendall_tau": tau, "spearman_rho": rho})

repeated_df = pd.DataFrame(records)
repeated_df.to_csv("results/tables/repeated_sampling.csv", index=False)

stability_summary = repeated_df.groupby("fraction")["kendall_tau"].agg(["mean", "std", "min", "max"]).reset_index()
stability_summary.to_csv("results/tables/stability_summary.csv", index=False)

confidence_intervals = (
    repeated_df[repeated_df["fraction"] < 1.0]
    .groupby("fraction")["kendall_tau"]
    .agg(mean="mean", lower_95=lambda x: x.quantile(0.025), upper_95=lambda x: x.quantile(0.975))
    .reset_index()
)
confidence_intervals.to_csv("results/tables/confidence_intervals.csv", index=False)
display(stability_summary)
display(confidence_intervals)


In [ ]:
# Track individual model rank stability (seed=123)
rng_ind = np.random.default_rng(INDIVIDUAL_SEED)
indiv_records = []
for frac in [0.25, 0.50, 0.75]:
    k = max(1, int(len(query_ids) * frac))
    for _ in range(n_repeats):
        sampled_q = rng_ind.choice(query_ids, size=k, replace=False)
        sub_rank = calculate_model_ranking(en_df[en_df["query_id"].isin(sampled_q)])
        for m in sub_rank.index:
            indiv_records.append({"fraction": frac, "model": m, "rank": sub_rank.loc[m, "rank"]})

indiv_df = pd.DataFrame(indiv_records).groupby(["fraction", "model"])["rank"].agg(["mean", "std"]).reset_index()
indiv_df.to_csv("results/tables/individual_model_stability.csv", index=False)
display(indiv_df.head(15))


## 4. Method 3 (M3): Cross-Lingual Rank Transfer Across 18 Languages
Evaluate model rankings independently across all 18 MIRAGE-Bench languages and compute Kendall's $\tau_b$ and Spearman's $\rho$ across all $\binom{18}{2} = 153$ language pairs.


In [ ]:
lang_rankings = {}
lang_summary_rows = []

for lang in LANGUAGES:
    ds_lang = load_dataset(DATASET_ID, lang)["dev"].to_pandas()
    df_lang = validate_pairwise_df(ds_lang)
    r_lang = calculate_model_ranking(df_lang)
    lang_rankings[lang] = r_lang
    lang_summary_rows.append({
        "language": lang,
        "records": len(df_lang),
        "queries": df_lang["query_id"].nunique(),
        "models": len(r_lang),
        "top_model": r_lang.index[0],
        "top_model_win_rate": r_lang.iloc[0]["win_rate"],
    })

lang_summary_df = pd.DataFrame(lang_summary_rows)
lang_summary_df.to_csv("results/tables/language_summary.csv", index=False)

corr_rows = []
for i in range(len(LANGUAGES)):
    for j in range(i + 1, len(LANGUAGES)):
        l1, l2 = LANGUAGES[i], LANGUAGES[j]
        r1, r2 = lang_rankings[l1], lang_rankings[l2]
        common = sorted(set(r1.index) & set(r2.index))
        s1 = [r1.loc[m, "win_rate"] for m in common]
        s2 = [r2.loc[m, "win_rate"] for m in common]
        corr_rows.append({
            "language_1": l1,
            "language_2": l2,
            "common_models": len(common),
            "kendall_tau": kendalltau(s1, s2, variant="b").statistic,
            "spearman_rho": spearmanr(s1, s2).statistic,
        })

lang_corr_df = pd.DataFrame(corr_rows)
lang_corr_df.to_csv("results/tables/language_correlations.csv", index=False)
display(lang_summary_df)
print(lang_corr_df[["kendall_tau", "spearman_rho"]].describe())
